In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# =========================
# Cell 1: Install (run once)
# =========================
!pip install -q pandas numpy scikit-learn tensorflow

In [4]:
# =========================
# Cell 2: Imports
# =========================
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
# =========================
# Cell 3: Load dataset
# =========================
DATA_PATH = "data.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(6810, 12)


,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [6]:
# =========================
# Cell 4: Clean + build searchable text
# =========================
def clean_text(x: str) -> str:
    x = "" if pd.isna(x) else str(x)
    x = x.lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

# Ensure columns exist (adjust if your dataset uses different names)
for col in ["isbn13", "title", "authors", "description"]:
    if col not in df.columns:
        df[col] = ""

# Clean fields
df["isbn13"] = df["isbn13"].astype(str).str.strip()

df["title"] = df["title"].apply(clean_text)
df["authors"] = df["authors"].apply(clean_text)
df["description"] = df["description"].apply(clean_text)

# Search field
df["search_text"] = (df["title"] + " " + df["authors"] + " " + df["description"]).str.strip()

# Drop empty text rows
df = df[df["search_text"].str.len() > 5].reset_index(drop=True)

print("Cleaned shape:", df.shape)
df[["isbn13", "title", "authors"]].head()

Cleaned shape: (6810, 13)


,isbn13,title,authors
0,9780002005883,gilead,marilynne robinson
1,9780002261982,spider's web,charles osborne;agatha christie
2,9780006163831,the one tree,stephen r. donaldson
3,9780006178736,rage of angels,sidney sheldon
4,9780006280897,the four loves,clive staples lewis


In [7]:
# =========================
# Cell 5: Train/Val split (book-disjoint split)
# =========================
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", train_df.shape, "Val:", val_df.shape)

# Optional: prove why train-only recall was 0
train_set = set(train_df["isbn13"].astype(str).str.strip())
val_set = set(val_df["isbn13"].astype(str).str.strip())
print("Val targets present in train:", len(train_set & val_set))

Train: (5448, 13) Val: (1362, 13)
Val targets present in train: 0


In [8]:
# =========================
# Cell 6: TF-IDF training on TRAIN (Correct)
# =========================
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

# Fit ONLY on train text
X_train = vectorizer.fit_transform(train_df["search_text"])
X_train = normalize(X_train)

# Build FULL corpus vectors using the same vectorizer
X_all = vectorizer.transform(df["search_text"])
X_all = normalize(X_all)

all_isbns = df["isbn13"].astype(str).str.strip().tolist()

print("X_train:", X_train.shape, "X_all:", X_all.shape)

X_train: (5448, 10000) X_all: (6810, 10000)


In [9]:
# =========================
# Cell 7: Build evaluation queries (from val)
# =========================
def make_queries(dataframe: pd.DataFrame):
    queries = []
    targets = []

    for _, row in dataframe.iterrows():
        isbn = str(row["isbn13"]).strip()

        q1 = row["title"]
        q2 = row["authors"]
        q3 = row["description"]
        q4 = (row["title"] + " " + row["authors"] + " " + row["description"]).strip()

        for q in [q1, q2, q3, q4]:
            if isinstance(q, str) and len(q.strip()) > 2:
                queries.append(q.strip())
                targets.append(isbn)

    return queries, targets

val_queries, val_targets = make_queries(val_df)

print("Validation queries:", len(val_queries))
print("Example query:", val_queries[0])
print("Target ISBN:", val_targets[0])

Validation queries: 5390
Example query: the firefly
Target ISBN: 9780312994815


In [10]:
# =========================
# Cell 8: Correct retrieval accuracy (Recall@K) on ALL corpus
# =========================
def recall_at_k_all(queries, targets, k=10):
    q_vec = vectorizer.transform(queries)
    q_vec = normalize(q_vec)

    sims = cosine_similarity(q_vec, X_all)  # (num_queries, num_all_books)

    # get top-k indices (unordered)
    topk_idx = np.argpartition(-sims, k-1, axis=1)[:, :k]

    hits = 0
    targets = [str(t).strip() for t in targets]

    for i in range(len(queries)):
        predicted_isbns = [all_isbns[j] for j in topk_idx[i]]
        hits += int(targets[i] in predicted_isbns)

    return hits / len(queries)

for k in [1, 5, 10]:
    r = recall_at_k_all(val_queries, val_targets, k=k)
    print(f"Baseline Recall@{k} (ALL corpus): {r:.4f}")

Baseline Recall@1 (ALL corpus): 0.6312
Baseline Recall@5 (ALL corpus): 0.7915
Baseline Recall@10 (ALL corpus): 0.8551


In [11]:
# =========================
# Cell 9: Real search function (Top-10 related books)
# =========================
def search_books(query, k=10):
    query = clean_text(query)
    q_vec = normalize(vectorizer.transform([query]))
    sims = cosine_similarity(q_vec, X_all).ravel()

    if k >= len(sims):
        top_idx = np.argsort(-sims)
    else:
        top_idx = np.argpartition(-sims, k-1)[:k]
        top_idx = top_idx[np.argsort(-sims[top_idx])]

    results = df.iloc[top_idx].copy()
    results["score"] = sims[top_idx]

    cols = [c for c in ["isbn13", "title", "authors", "score"] if c in results.columns]
    return results[cols]

# Example:
print(search_books("harry potter magic wizard school", k=10).to_string(index=False))

       isbn13                                                      title                                 authors    score
9781932100594                          mapping the world of harry potter             mercedes lackey;leah wilson 0.556747
9780439682589                                               harry potter                           j. k. rowling 0.523293
9780755311514                                the science of harry potter                         roger highfield 0.503475
9780439827607                                the harry potter collection                           j. k. rowling 0.502454
9780972393614 ultimate unofficial guide to the mysteries of harry potter       galadriel waters;astre mithrandir 0.500216
9780439554893                    harry potter and the chamber of secrets             j. k. rowling;mary grandpre 0.499425
9780747546245                        harry potter and the goblet of fire                           j. k. rowling 0.491877
9780812694550           

In [12]:
###################------10-------#########


# We'll train on train_df using stronger queries

train_queries = []
train_docs = []

for _, r in train_df.iterrows():
    title = str(r["title"]).strip()
    author = str(r["authors"]).strip()
    desc = str(r["description"]).strip()

    # Use first 30 words of description
    desc_short = " ".join(desc.split()[:30]).strip()

    # Stronger query (like real user input)
    q = f"{title} {author} {desc_short}".strip()

    train_queries.append(q)
    train_docs.append(str(r["search_text"]))

# TF-IDF vectors
Q = normalize(vectorizer.transform(train_queries)).astype(np.float32)
D = normalize(vectorizer.transform(train_docs)).astype(np.float32)

print(Q.shape, D.shape)

(5448, 10000) (5448, 10000)


In [13]:
# =========================
# Cell 11: Build TensorFlow Dataset
# =========================
import tensorflow as tf

BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

# Convert sparse -> dense (for NN). For big datasets, this uses RAM.
# If RAM is low, reduce max_features or batch size.
Q_dense = Q.toarray().astype(np.float32)
D_dense = D.toarray().astype(np.float32)

ds = tf.data.Dataset.from_tensor_slices((Q_dense, D_dense))
ds = ds.shuffle(20000, seed=42).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [14]:
# =========================
# Cell 11: Neural embedding model
# =========================
import tensorflow as tf

FEATURES = Q_dense.shape[1]  # 10000

def make_encoder():
    inp = tf.keras.Input(shape=(FEATURES,), name="tfidf_input")
    x = tf.keras.layers.Dense(512, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dense(128)(x)

    # ✅ Wrap TF op inside a Keras Lambda layer
    out = tf.keras.layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1), name="l2norm")(x)

    return tf.keras.Model(inp, out)

query_encoder = make_encoder()
doc_encoder   = make_encoder()

query_encoder.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ tfidf_input (InputLayer)        │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     5,120,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ l2norm (Lambda)                 │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,284,736 (20.16 MB)

 Trainable params: 5,284,736 (20.16 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# =========================
# Cell 12: InfoNCE loss training loop
# =========================
optimizer = tf.keras.optimizers.Adam(1e-3)

@tf.function
def train_step(q_batch, d_batch):
    with tf.GradientTape() as tape:
        q_emb = query_encoder(q_batch, training=True)  # (B, 128)
        d_emb = doc_encoder(d_batch, training=True)    # (B, 128)

        # Similarity matrix (B x B)
        logits = tf.matmul(q_emb, d_emb, transpose_b=True)

        # Labels: diagonal is correct match
        labels = tf.range(tf.shape(logits)[0])
        loss = tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=logits)
        )

    vars_ = query_encoder.trainable_variables + doc_encoder.trainable_variables
    grads = tape.gradient(loss, vars_)
    optimizer.apply_gradients(zip(grads, vars_))
    return loss

EPOCHS = 10
for epoch in range(EPOCHS):
    losses = []
    for qb, db in ds:
        l = train_step(qb, db)
        losses.append(float(l.numpy()))
    print(f"Epoch {epoch+1}/{EPOCHS} loss: {np.mean(losses):.4f}")

Epoch 1/10 loss: 4.6589
Epoch 2/10 loss: 4.2060
Epoch 3/10 loss: 4.0931
Epoch 4/10 loss: 4.0361
Epoch 5/10 loss: 3.9990
Epoch 6/10 loss: 3.9764
Epoch 7/10 loss: 3.9689
Epoch 8/10 loss: 3.9617
Epoch 9/10 loss: 3.9525
Epoch 10/10 loss: 3.9526


In [16]:
# Build stronger validation queries
def make_queries_stronger(df_in, max_desc_words=30):
    queries, targets = [], []

    for _, r in df_in.iterrows():
        isbn = str(r["isbn13"]).strip()
        title = str(r["title"]).strip()
        author = str(r["authors"]).strip()
        desc = str(r["description"]).strip()

        desc_short = " ".join(desc.split()[:max_desc_words]).strip()

        q = f"{title} {author} {desc_short}".strip()

        if len(q) > 5:
            queries.append(q)
            targets.append(isbn)

    return queries, targets

val_queries, val_targets = make_queries_stronger(val_df)

# Embed ALL documents
D_all_tfidf = normalize(vectorizer.transform(df["search_text"])).toarray().astype(np.float32)
all_doc_emb = doc_encoder(D_all_tfidf, training=False).numpy().astype(np.float32)

# Embed validation queries
val_q_tfidf = normalize(vectorizer.transform(val_queries)).toarray().astype(np.float32)
val_q_emb = query_encoder(val_q_tfidf, training=False).numpy().astype(np.float32)

# Similarity
sims_all = val_q_emb @ all_doc_emb.T

all_isbns = df["isbn13"].astype(str).str.strip().tolist()

def recall_at_k_emb_all(sims, targets, k=10):
    topk_idx = np.argpartition(-sims, k-1, axis=1)[:, :k]
    hits = 0
    for i in range(len(targets)):
        predicted = [all_isbns[j] for j in topk_idx[i]]
        hits += int(targets[i] in predicted)
    return hits / len(targets)

for k in [1, 5, 10, 15, 20]:
    print(f"Neural Recall@{k}:", recall_at_k_emb_all(sims_all, val_targets, k))

Neural Recall@1: 0.486784140969163
Neural Recall@5: 0.6688693098384728
Neural Recall@10: 0.7334801762114538
Neural Recall@15: 0.7687224669603524
Neural Recall@20: 0.7966226138032305


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# Precompute TF-IDF for all books
X_all = normalize(vectorizer.transform(df["search_text"]))

def hybrid_search(query, k=10, pre_k=200):
    # Step 1: TF-IDF shortlist
    q_tfidf = normalize(vectorizer.transform([query]))
    sims_tfidf = cosine_similarity(q_tfidf, X_all).ravel()

    idx = np.argpartition(-sims_tfidf, pre_k-1)[:pre_k]
    idx = idx[np.argsort(-sims_tfidf[idx])]

    # Step 2: Neural rerank
    q_dense = q_tfidf.toarray().astype(np.float32)
    q_emb = query_encoder(q_dense, training=False).numpy().astype(np.float32)

    doc_emb_subset = all_doc_emb[idx]
    sims_nn = (q_emb @ doc_emb_subset.T).ravel()

    top = np.argsort(-sims_nn)[:k]
    final_idx = idx[top]

    out = df.iloc[final_idx].copy()
    out["score"] = sims_nn[top]

    cols = [c for c in ["isbn13", "title", "authors", "score"] if c in out.columns]
    return out[cols]

# Test
print(hybrid_search("the firefly", k=10).to_string(index=False))

       isbn13                                      title          authors    score
9784770027726                                   ゲンジモノガタリ              紫式部 0.971977
9780060542382 amelia bedelia 40th anniversary collection     peggy parish 0.601554
9780060511142                   teach us, amelia bedelia     peggy parish 0.595980
9780060510862                            the forever war     joe haldeman 0.594972
9780007158515              i can read with me eyes shut!        dr. seuss 0.592675
9780060546571                          three rotten eggs  gregory maguire 0.584952
9780060511111                   amelia bedelia helps out     peggy parish 0.563405
9780060092573                           the terminal man michael crichton 0.549541
9780007150304                 beware, princess elizabeth    carolyn meyer 0.537993
9780060511067                amelia bedelia goes camping     peggy parish 0.508815


In [18]:
import pickle
import numpy as np

SAVE_PATH = "neural_book_search_full.pkl"

# Ensure these columns exist
for col in ["isbn13", "title", "authors", "description", "thumbnail"]:
    if col not in df.columns:
        df[col] = ""

# Clean
df["isbn13"] = df["isbn13"].astype(str).str.strip()
df["title"] = df["title"].astype(str)
df["authors"] = df["authors"].astype(str)
df["description"] = df["description"].astype(str)
df["thumbnail"] = df["thumbnail"].astype(str)

# IMPORTANT: embeddings must match df order
all_isbns = df["isbn13"].tolist()

meta = {
    "title": df["title"].tolist(),
    "authors": df["authors"].tolist(),
    "description": df["description"].tolist(),
    "thumbnail": df["thumbnail"].tolist()
}

data_to_store = {
    "book_ids": all_isbns,
    "doc_embeddings": all_doc_emb.astype(np.float32),   # from your Cell 14
    "tfidf_model": vectorizer,
    "query_encoder_weights": query_encoder.get_weights(),
    "doc_encoder_weights": doc_encoder.get_weights(),
    "max_features": int(FEATURES),
    "meta": meta
}

with open(SAVE_PATH, "wb") as f:
    pickle.dump(data_to_store, f)

print("✅ Saved:", SAVE_PATH)
print("Books:", len(all_isbns))
print("Embeddings:", all_doc_emb.shape)

✅ Saved: neural_book_search_full.pkl
Books: 6810
Embeddings: (6810, 128)
